In [3]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import ElementClickInterceptedException
from selenium.common.exceptions import StaleElementReferenceException, NoSuchElementException
import time
from utils import writeJson, readJson
import os
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.keys import Keys
import json
import re
from bs4 import BeautifulSoup as soup
import datetime
from datetime import datetime as dt
from tqdm import tqdm

In [112]:
def getPlayerStats(driver, role):
    data = {}
    roles = getRoles(role)
    found = False
    for r in roles:
        try: 
            table = driver.find_element(By.XPATH, f'//*[@id="scout_full_{r}"]/tbody')
            found = True
        except:
            continue
    
    if not found:
        return {}

    for row in table.find_elements(By.TAG_NAME, 'tr'):
        th = row.find_element(By.TAG_NAME, 'th')
        data_desc = th.get_attribute('data-tip')
        tds = row.find_elements(By.TAG_NAME, 'td')
        if len(tds) > 0 and th.text != '':
            value, perc = tds[0].text , tds[1].text
            #f'{th.text} ({data_desc})'
            data[th.text] = value

    return data

def getPlayerAnag(driver):
    anag_div = driver.find_element(By.XPATH, '/html/body/div[4]/div[3]/div[1]/div[2]')
    player = anag_div.find_element(By.TAG_NAME, 'h1').text

    anag_elem1 = driver.find_element(By.XPATH,'//*[@id="meta"]/div[2]/p[1]').text
    anag_elem2 = driver.find_element(By.XPATH,'//*[@id="meta"]/div[2]/p[2]').text
    anag_elem3 = driver.find_element(By.XPATH,'//*[@id="meta"]/div[2]/p[3]').text
    anag_elem4 = driver.find_element(By.XPATH,'//*[@id="meta"]/div[2]/p[4]').text
    anag_elem5 = driver.find_element(By.XPATH,'//*[@id="meta"]/div[2]/p[5]').text

    anag_elem1_split = anag_elem1.split('▪')
    anag_elem2_split = anag_elem2.split(' ')
    position = anag_elem1_split[0].split(':')[1].strip()
    #footed = anag_elem1_split[1].split(':')[1].strip()
    height, weight = anag_elem2_split[0], anag_elem2_split[1]
    year_birth = anag_elem3.split(' ')[3]
    nat = anag_elem4.split(' ')[2]
    team = anag_elem5.split(' ')[1]
    height,weight,year_birth, nat, team
    return dict(player=player, 
                position=position, 
                #foot=footed, 
                height= height, 
                weight=weight, 
                year_birth=year_birth, 
                nationality=nat, team=team)


def getPlayerRecord(driver, url):
    driver.get(url)
    player = getPlayerAnag(driver)
    stats = getPlayerStats(driver, player['position'])
    player['stats'] = stats
    return player

def getRoles(role):
    rr=[]
    if '(' in role:
        roles_split = role.split('(')
    else:
        roles_split = [role]

    for r in roles_split:
        if '-' in r:
            r_split = r.split('-')
            rr.append(r_split[0])
            rl = r_split[1]
            if rl[-1] == ')':
                rl = rl[:-1]
            rr.append(rl)

        elif ',' in r:
            rr.append(r.split(',')[0])
        else:
            rr.append(r.strip())


    return rr

In [4]:
url = 'https://fbref.com/en/players/20730eae/scout/12229/Rafael-Leao-Scouting-Report'


In [108]:
driver = webdriver.Chrome()
url = 'https://fbref.com/en/'
driver.get(url)

In [109]:
#cookie_button = driver.find_element(By.XPATH, '//*[@id="bd44e2e6-0aae-43e0-bae8-847a8a5e55a6"]/div[2]/button[2]')
cookie_button = driver.find_elements(By.TAG_NAME, 'button')

#//*[@id="10b57b08-c511-465a-b19d-f1569498078c"]/div[2]/button[2]
#//*[@id="5363d468-fad8-45fc-8284-ea0f1779dec5"]/div[2]/button[2]

for b in cookie_button:
    if b.text == 'Accetta tutto':
        b.click()


In [116]:
#records = []
urls = [
    #'https://fbref.com/en/players/20730eae/scout/12229/Rafael-Leao-Scouting-Report',
    #'https://fbref.com/en/players/d4c9725f/scout/12229/Theo-Hernandez-Scouting-Report',
    #'https://fbref.com/en/players/a0d55a09/scout/12207/Bradley-Barcola-Scouting-Report',
    #'https://fbref.com/en/players/1f44ac21/scout/12192/Erling-Haaland-Scouting-Report',
    'https://fbref.com/en/players/0db169ae/scout/11611/Sandro-Tonali-Scouting-Report',
    'https://fbref.com/en/players/3f5f38fb/scout/12229/Nicolo-Casale-Scouting-Report',
    'https://fbref.com/en/players/afb61630/scout/12229/Tijjani-Reijnders-Scouting-Report'
]
for url in urls:
    print(url)
    records.append(getPlayerRecord(driver,url))


https://fbref.com/en/players/0db169ae/scout/11611/Sandro-Tonali-Scouting-Report
https://fbref.com/en/players/3f5f38fb/scout/12229/Nicolo-Casale-Scouting-Report
https://fbref.com/en/players/afb61630/scout/12229/Tijjani-Reijnders-Scouting-Report


In [118]:
writeJson(records,'Dataset/Fbref/prova_records.json')

In [105]:
roles = ['MF (CM-DM)','FW-MF (AM, left)','DF (FB, left)', 'FW-MF']
def getRoles(role):
    rr=[]
    if '(' in role:
        roles_split = role.split('(')
    else:
        roles_split = [role]

    for r in roles_split:
        if '-' in r:
            r_split = r.split('-')
            rr.append(r_split[0])
            rl = r_split[1]
            if rl[-1] == ')':
                rl = rl[:-1]
            rr.append(rl)

        elif ',' in r:
            rr.append(r.split(',')[0])
        else:
            rr.append(r.strip())


    return rr

getRoles(roles[1])



['FW', 'MF ', 'AM']

In [121]:
p = records[0]


In [ ]:
player_name = p['player']
prompt = f""""You are a professional football scout with expertise in analyzing players' technical and tactical characteristics. 
        I need you to generate a detailed report for a player, based on the provided list of statistics that describe their performance averaged per 90 minutes. 
        Your task is to analyze this data and provide a report as follows:

        ### Input Data:
            - Player: {player_name}
            - Position: {p['position']}
            - Year birth: {p['year_birth']}
            - Height: {p['height']}
            - Weight: {p['weight']}
            - Statistics per 90 minutes: 
            {p['stats']}

        ### Output Format:
        Your report should be structured in the following way:
        **Player**: {player_name}
        **Strengths**: 
        Highlight the player's key strengths evident from their playing style.
        **Weaknesses**: 
        Point out areas where the player needs improvement.
        **Summary**:
        A brief summary of the player's overall performance.


        ### Notes for Analysis:
        - Use concise and professional language.
        - The report should be realistic for scouting purposes.
        - Do not generate code or class structures. Focus only on the football analysis.
        - The output must be in plain text, clearly formatted according to the structure above.
        - Do not write the name of the player 
        - Do not include statistics into report
        
        Provide the report below this prompt, clearly labeled as "Generated Report".
        ###Generated Report:
        """
prompt

'"You are a professional football scout with expertise in analyzing players\' technical and tactical characteristics. \n        I need you to generate a detailed report for a player, based on the provided list of statistics that describe their performance averaged per 90 minutes. \n        Your task is to analyze this data and provide a report as follows:\n\n        ### Input Data:\n            - Player: Rafael Leão\n            - Position: FW-MF (AM, left)\n            - Year birth: 1999\n            - Height: 189cm,\n            - Weight: 78kg\n            - Statistics per 90 minutes: \n            {\'Goals\': \'0.32\', \'Assists\': \'0.32\', \'Goals + Assists\': \'0.64\', \'Non-Penalty Goals\': \'0.32\', \'Penalty Kicks Made\': \'0.00\', \'Penalty Kicks Attempted\': \'0.00\', \'Yellow Cards\': \'0.18\', \'Red Cards\': \'0.00\', \'xG: Expected Goals\': \'0.36\', \'npxG: Non-Penalty xG\': \'0.36\', \'xAG: Exp. Assisted Goals\': \'0.33\', \'npxG + xAG\': \'0.68\', \'Progressive Carries